In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/ipythonx/mvtec-ad/readme.txt
/kaggle/input/datasets/ipythonx/mvtec-ad/license.txt
/kaggle/input/datasets/ipythonx/mvtec-ad/wood/readme.txt
/kaggle/input/datasets/ipythonx/mvtec-ad/wood/license.txt
/kaggle/input/datasets/ipythonx/mvtec-ad/wood/ground_truth/hole/000_mask.png
/kaggle/input/datasets/ipythonx/mvtec-ad/wood/ground_truth/hole/003_mask.png
/kaggle/input/datasets/ipythonx/mvtec-ad/wood/ground_truth/hole/004_mask.png
/kaggle/input/datasets/ipythonx/mvtec-ad/wood/ground_truth/hole/005_mask.png
/kaggle/input/datasets/ipythonx/mvtec-ad/wood/ground_truth/hole/001_mask.png
/kaggle/input/datasets/ipythonx/mvtec-ad/wood/ground_truth/hole/006_mask.png
/kaggle/input/datasets/ipythonx/mvtec-ad/wood/ground_truth/hole/008_mask.png
/kaggle/input/datasets/ipythonx/mvtec-ad/wood/ground_truth/hole/007_mask.png
/kaggle/input/datasets/ipythonx/mvtec-ad/wood/ground_truth/hole/002_mask.png
/kaggle/input/datasets/ipythonx/mvtec-ad/wood/ground_truth/hole/009_mask.png
/kaggle/in

In [2]:
from pathlib import Path
import torch

REPO_ROOT = Path("/kaggle/working/trajlik-anomaly-detection")
INVAD_ROOT = REPO_ROOT / "baseline" / "InversionAD"

SOURCE_MODEL_DIR = Path(
    "/kaggle/input/models/gwyser/invad-mvtecad-dit-gigant/"
    "pytorch/default/1/MVTecAD-DiT_Gigant/"
    "results/dit_gigant_mvtecad"
)

SOURCE_CHECKPOINT = SOURCE_MODEL_DIR / "model.pth"
SOURCE_CONFIG = SOURCE_MODEL_DIR / "config.yaml"

DATA_ROOT = Path("/kaggle/input/datasets/ipythonx/mvtec-ad")

WORK_MODEL_DIR = Path("/kaggle/working/invad_official_nfe3")
RUNTIME_CHECKPOINT = WORK_MODEL_DIR / "model.pth"
RUNTIME_CONFIG = WORK_MODEL_DIR / "config.yaml"

LOG_PATH = Path("/kaggle/working/invad_baseline_nfe3.log")
OUTPUT_JSON = Path("/kaggle/working/invad_baseline_nfe3.json")

print("Checkpoint:", SOURCE_CHECKPOINT)
print("Config:", SOURCE_CONFIG)
print("Dataset:", DATA_ROOT)

Checkpoint: /kaggle/input/models/gwyser/invad-mvtecad-dit-gigant/pytorch/default/1/MVTecAD-DiT_Gigant/results/dit_gigant_mvtecad/model.pth
Config: /kaggle/input/models/gwyser/invad-mvtecad-dit-gigant/pytorch/default/1/MVTecAD-DiT_Gigant/results/dit_gigant_mvtecad/config.yaml
Dataset: /kaggle/input/datasets/ipythonx/mvtec-ad


In [3]:
import subprocess

if not REPO_ROOT.exists():
    subprocess.run(
        [
            "git",
            "clone",
            "https://github.com/phanthehoang2503/trajlik-anomaly-detection.git",
            str(REPO_ROOT),
        ],
        check=True,
    )
else:
    subprocess.run(
        ["git", "-C", str(REPO_ROOT), "pull", "--ff-only"],
        check=True,
    )

print("Repo:", REPO_ROOT)

Cloning into '/kaggle/working/trajlik-anomaly-detection'...


Repo: /kaggle/working/trajlik-anomaly-detection


In [4]:
os.chdir('/kaggle/working/trajlik-anomaly-detection')
%pip install -q -r requirements.txt

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.7/57.7 kB 1.5 MB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.6/51.6 kB 2.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.8/60.8 kB 2.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.9/89.9 kB 3.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.1/80.1 kB 3.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 906.4/906.4 MB 1.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.2/7.2 MB 11.9 MB/s eta 0:00:0000:0100:01m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.5/78.5 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 62.2 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 101.7/101.7 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 62.6 MB/s eta 0:00:0000:01:00:01
   

In [5]:
!mkdir -p /kaggle/working/invad_pretrained
!cp /kaggle/input/models/gwyser/invad-mvtecad-dit-gigant/pytorch/default/1/MVTecAD-DiT_Gigant/results/dit_gigant_mvtecad/model.pth /kaggle/working/invad_pretrained/
!cp /kaggle/input/models/gwyser/invad-mvtecad-dit-gigant/pytorch/default/1/MVTecAD-DiT_Gigant/results/dit_gigant_mvtecad/config.yaml /kaggle/working/invad_pretrained/

In [6]:
import yaml

config_path = "/kaggle/working/invad_pretrained/config.yaml"

with open(config_path) as f:
    config = yaml.safe_load(f)

config["data"]["data_root"] = "/kaggle/input/datasets/ipythonx/mvtec-ad"
config["data"]["batch_size"] = 1
config["data"]["num_workers"] = 2
config["data"]["pin_memory"] = True
config["meta"]["device"] = "cuda"

with open(config_path, "w") as f:
    yaml.safe_dump(config, f, sort_keys=False)

In [7]:
%cd /kaggle/working/trajlik-anomaly-detection/baseline/InversionAD

!PYTHONPATH=. python -m src.evaluate \
    --eval_strategy inversion \
    --checkpoint_path /kaggle/working/invad_pretrained/model.pth \
    --eval_step 3

/kaggle/working/trajlik-anomaly-detection/baseline/InversionAD
INFO:matplotlib.font_manager:generated new fontManager
INFO:root:Using multi-class dataset
INFO:root:Using feature space reconstruction with efficientnet-b4 backbone
INFO:global_logger: not exist, load from https://github.com/lukemelas/EfficientNet-PyTorch/releases/download/1.0/efficientnet-b4-6ed6700e.pth
Downloading: "https://github.com/lukemelas/EfficientNet-PyTorch/releases/download/1.0/efficientnet-b4-6ed6700e.pth" to /root/.cache/torch/hub/checkpoints/efficientnet-b4-6ed6700e.pth
100%|██████████████████████████████████████| 74.4M/74.4M [00:01<00:00, 73.5MB/s]
INFO:global_logger:Loaded ImageNet pretrained efficientnet-b4
INFO:root:Loaded model from /kaggle/working/invad_pretrained/model.pth
INFO:root:Number of parameters in the model: 1223.83M
INFO:root:[metal_nut] Evaluating on 93 anomalous samples and 22 normal samples
INFO:root:[metal_nut] Evaluation step: 3
INFO:root:[metal_nut] Epoch: Eval
100%|███████████████████